### LGC score for K = 25 case

In [3]:
import os
import json
import torch
import torch.nn as nn
from torch.func import vmap, functional_call, jacrev
import numpy as np

# Set fixed seed for reproducibility across all feature runs
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

class LinearProbe(nn.Module):
    def __init__(self, input_dim: int, num_classes: int = 6):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

def compute_linear_lgc(features: np.ndarray, labels: np.ndarray, num_classes: int = 6):
    N, D = features.shape
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Normalize features (L2 sphere)
    norms = np.linalg.norm(features, axis=1, keepdims=True) + 1e-8
    features_norm = torch.tensor(features / norms, dtype=torch.float32, device=device)

    # 2. Instantiate Linear Model with fixed seed
    torch.manual_seed(SEED)
    model = LinearProbe(input_dim=D, num_classes=num_classes).to(device)
    model.eval()

    params = dict(model.named_parameters())
    buffers = dict(model.named_buffers())

    # Single-sample logit function: (D,) -> (C,)
    def predict_single(p, b, x_vec):
        return functional_call(model, (p, b), x_vec.unsqueeze(0)).squeeze(0)

    # Vectorized per-sample Jacobian w.r.t model parameters
    # jacrev produces dict of tensors with shape (N, C, *param_shape)
    sample_jac_fn = vmap(jacrev(predict_single, argnums=0), in_dims=(None, None, 0))
    jac_dict = sample_jac_fn(params, buffers, features_norm)

    # Flatten and concatenate parameters into (N, C * P)
    flat_jacs = [g.reshape(N, -1) for g in jac_dict.values()]
    J = torch.cat(flat_jacs, dim=1)  # Shape: (N, Total_Grad_Dim)

    # Compute empirical NTK: Theta = J @ J.T (N x N)
    Theta = (J @ J.T).detach().cpu().numpy()

    # 3. Kernel Centering
    H = np.eye(N) - (1.0 / N) * np.ones((N, N))
    Theta_tilde = H @ Theta @ H

    Y_onehot = np.eye(num_classes)[labels]
    K_Y = Y_onehot @ Y_onehot.T
    K_Y_tilde = H @ K_Y @ H

    norm_Theta = np.linalg.norm(Theta_tilde, 'fro') + 1e-12
    norm_KY = np.linalg.norm(K_Y_tilde, 'fro') + 1e-12

    # 4. Class-wise and Overall LGC
    class_scores = {}
    total_alignment = 0.0

    for c in range(num_classes):
        y_c = (labels == c).astype(np.float64)
        y_c_tilde = y_c - np.mean(y_c)
        
        # Intra-class gradient correlation
        intra_c = float(y_c_tilde.T @ Theta_tilde @ y_c_tilde)
        score_c = intra_c / (norm_Theta * norm_KY)
        
        class_scores[int(c)] = score_c
        total_alignment += score_c

    return float(total_alignment), class_scores

# --- Pipeline Execution ---
base_dir = "../model_embeddings"
output_dir = "../model_scores/LGC_linear"
os.makedirs(output_dir, exist_ok=True)

results = {}

for model_name in sorted(os.listdir(base_dir)):
    model_path = os.path.join(base_dir, model_name)
    if not os.path.isdir(model_path):
        continue
        
    npz_path = os.path.join(model_path, "features.npz")
    if not os.path.exists(npz_path):
        continue
        
    print(f"Calculating Linear LGC for {model_name}...")
    data = np.load(npz_path)
    features = data["embeddings"]
    labels = data["labels"]
    
    overall, class_wise = compute_linear_lgc(features, labels, num_classes=6)
    
    results[model_name] = {
        "overall_lgc": overall,
        "class_wise_lgc": class_wise
    }

output_file = os.path.join(output_dir, "all_models_lgc.json")
with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"\nSaved Linear NTK-LGC scores to {output_file}")

Calculating Linear LGC for AVES-bio...
Calculating Linear LGC for BirdAVES-biox-base...
Calculating Linear LGC for DistilHuBERT...
Calculating Linear LGC for HuBERT...
Calculating Linear LGC for MusicHuBERT...
Calculating Linear LGC for WavLM...

Saved Linear NTK-LGC scores to ../model_scores/LGC_linear\all_models_lgc.json


### Model LGC - scores for K shot

In [1]:
import os
import json
import torch
import torch.nn as nn
from torch.func import vmap, functional_call, jacrev
import numpy as np

In [4]:

# Set fixed seed for reproducibility across all feature runs
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

class LinearProbe(nn.Module):
    def __init__(self, input_dim: int, num_classes: int = 6):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

def compute_linear_lgc(features: np.ndarray, labels: np.ndarray, num_classes: int = 6):
    N, D = features.shape
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Normalise features (L2 sphere)
    norms = np.linalg.norm(features, axis=1, keepdims=True) + 1e-8
    features_norm = torch.tensor(features / norms, dtype=torch.float32, device=device)

    # 2. Instantiate Linear Model with fixed seed
    torch.manual_seed(SEED)
    model = LinearProbe(input_dim=D, num_classes=num_classes).to(device)
    model.eval()

    params = dict(model.named_parameters())
    buffers = dict(model.named_buffers())

    # Single-sample logit function: (D,) -> (C,)
    def predict_single(p, b, x_vec):
        return functional_call(model, (p, b), x_vec.unsqueeze(0)).squeeze(0)

    # Vectorized per-sample Jacobian w.r.t model parameters
    # jacrev produces dict of tensors with shape (N, C, *param_shape)
    sample_jac_fn = vmap(jacrev(predict_single, argnums=0), in_dims=(None, None, 0))
    jac_dict = sample_jac_fn(params, buffers, features_norm)

    # Flatten and concatenate parameters into (N, C * P)
    flat_jacs = [g.reshape(N, -1) for g in jac_dict.values()]
    J = torch.cat(flat_jacs, dim=1)  # Shape: (N, Total_Grad_Dim)

    # Compute empirical NTK: Theta = J @ J.T (N x N)
    Theta = (J @ J.T).detach().cpu().numpy()

    # 3. Kernel Centering
    H = np.eye(N) - (1.0 / N) * np.ones((N, N))
    Theta_tilde = H @ Theta @ H

    Y_onehot = np.eye(num_classes)[labels]
    K_Y = Y_onehot @ Y_onehot.T
    K_Y_tilde = H @ K_Y @ H

    norm_Theta = np.linalg.norm(Theta_tilde, 'fro') + 1e-12
    norm_KY = np.linalg.norm(K_Y_tilde, 'fro') + 1e-12

    # 4. Class-wise and Overall LGC
    class_scores = {}
    total_alignment = 0.0

    for c in range(num_classes):
        y_c = (labels == c).astype(np.float64)
        y_c_tilde = y_c - np.mean(y_c)
        
        # Intra-class gradient correlation
        intra_c = float(y_c_tilde.T @ Theta_tilde @ y_c_tilde)
        score_c = intra_c / (norm_Theta * norm_KY)
        
        class_scores[int(c)] = score_c
        total_alignment += score_c

    return float(total_alignment), class_scores


def get_first_k_subset(features: np.ndarray, labels: np.ndarray, shots_per_class: int):
    """
    Deterministically selects the first `shots_per_class` samples for each class
    in the order they appear in the archive.
    """
    selected_indices = []

    for c in np.unique(labels):
        c_indices = np.where(labels == c)[0]
        if len(c_indices) < shots_per_class:
            raise ValueError(f"Class {c} only has {len(c_indices)} samples; cannot pick {shots_per_class}.")
        
        selected_indices.extend(c_indices[:shots_per_class])

    selected_indices = np.array(selected_indices)
    return features[selected_indices], labels[selected_indices]


# --- Configuration ---
SHOTS_PER_CLASS = 1   # Set to 10, 5, or 1

base_dir = "../model_embeddings"
output_dir = f"../model_scores_{SHOTS_PER_CLASS}_shot/LGC_linear"
os.makedirs(output_dir, exist_ok=True)

results = {}

for model_name in sorted(os.listdir(base_dir)):
    model_path = os.path.join(base_dir, model_name)
    if not os.path.isdir(model_path):
        continue
        
    npz_path = os.path.join(model_path, "features.npz")
    if not os.path.exists(npz_path):
        continue
        
    print(f"Calculating Linear LGC for {model_name} on first {SHOTS_PER_CLASS} samples/class...")
    data = np.load(npz_path)
    full_features = data["embeddings"]
    full_labels = data["labels"]
    
    # Deterministic subset extraction
    features, labels = get_first_k_subset(
        full_features, 
        full_labels, 
        shots_per_class=SHOTS_PER_CLASS
    )
    
    overall, class_wise = compute_linear_lgc(features, labels, num_classes=6)
    
    results[model_name] = {
        "shots_per_class": SHOTS_PER_CLASS,
        "total_samples": int(features.shape[0]),
        "overall_lgc": overall,
        "class_wise_lgc": class_wise
    }

output_file = os.path.join(output_dir, "all_models_lgc.json")
with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"\nSaved deterministic {SHOTS_PER_CLASS}-shot Linear NTK-LGC scores to {output_file}")

Calculating Linear LGC for AVES-bio on first 1 samples/class...
Calculating Linear LGC for BirdAVES-biox-base on first 1 samples/class...
Calculating Linear LGC for DistilHuBERT on first 1 samples/class...
Calculating Linear LGC for HuBERT on first 1 samples/class...
Calculating Linear LGC for MusicHuBERT on first 1 samples/class...
Calculating Linear LGC for WavLM on first 1 samples/class...

Saved deterministic 1-shot Linear NTK-LGC scores to ../model_scores_1_shot/LGC_linear\all_models_lgc.json


### Train the model with L_2 regularized loss

In [1]:
import os
import json
import torch
import torch.nn as nn
import numpy as np

In [2]:

# Set fixed seed for deterministic initialization and training
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

class LinearProbe(nn.Module):
    def __init__(self, input_dim: int, num_classes: int = 6):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

def get_first_k_subset(features: np.ndarray, labels: np.ndarray, shots_per_class: int):
    """
    Deterministically selects the first `shots_per_class` samples for each class
    in the order they appear in the archive.
    """
    selected_indices = []

    for c in np.unique(labels):
        c_indices = np.where(labels == c)[0]
        if len(c_indices) < shots_per_class:
            raise ValueError(f"Class {c} only has {len(c_indices)} samples; cannot pick {shots_per_class}.")
        selected_indices.extend(c_indices[:shots_per_class])

    selected_indices = np.array(selected_indices)
    return features[selected_indices], labels[selected_indices]

def train_linear_model(
    features: np.ndarray, 
    labels: np.ndarray, 
    num_classes: int = 6, 
    epochs: int = 100, 
    lr: float = 0.01, 
    weight_decay: float = 0.05
):
    """
    Trains a linear probe via full-batch gradient descent with L2 weight decay.
    """
    N, D = features.shape
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Unit-norm feature normalisation to bound input magnitudes
    norms = np.linalg.norm(features, axis=1, keepdims=True) + 1e-8
    features_norm = torch.tensor(features / norms, dtype=torch.float32, device=device)
    labels_tensor = torch.tensor(labels, dtype=torch.long, device=device)

    # 2. Model Initialisation
    torch.manual_seed(SEED)
    model = LinearProbe(input_dim=D, num_classes=num_classes).to(device)
    model.train()

    criterion = nn.CrossEntropyLoss()
    # Explicit L2 regularisation applied directly through optimizer weight decay
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # 3. Full-Batch Training Loop
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        logits = model(features_norm)
        loss = criterion(logits, labels_tensor)
        loss.backward()
        optimizer.step()

    # Final training metrics
    model.eval()
    with torch.no_grad():
        final_logits = model(features_norm)
        final_loss = criterion(final_logits, labels_tensor).item()
        preds = torch.argmax(final_logits, dim=1)
        final_acc = (preds == labels_tensor).float().mean().item()

    return model, final_loss, final_acc

# --- Configuration ---
SHOTS_PER_CLASS = 10   # Set to 25, 10, 5, or 1
NUM_CLASSES = 6
NUM_EPOCHS = 100
LEARNING_RATE = 0.01

# L2 regularisation scaling inversely with sample budget K
LAMBDA_MAP = {
    1: 0.5,
    5: 0.1,
    10: 0.05,
    25: 0.01
}
WEIGHT_DECAY = LAMBDA_MAP.get(SHOTS_PER_CLASS, 0.05)

# Adjust path based on execution inside `models_classification/` or workspace root[cite: 1]
base_dir = "../model_embeddings" if os.path.exists("../model_embeddings") else "./model_embeddings"[cite: 1]
weights_output_dir = f"../trained_models_{SHOTS_PER_CLASS}_shot/Linear" if os.path.exists("../model_embeddings") else f"./trained_models_{SHOTS_PER_CLASS}_shot/Linear"
os.makedirs(weights_output_dir, exist_ok=True)

training_summary = {}

for model_name in sorted(os.listdir(base_dir)):
    model_path = os.path.join(base_dir, model_name)
    if not os.path.isdir(model_path):
        continue

    npz_path = os.path.join(model_path, "features.npz")
    if not os.path.exists(npz_path):
        continue

    print(f"Training Linear Model for {model_name} ({SHOTS_PER_CLASS}-shot, λ={WEIGHT_DECAY})...")
    data = np.load(npz_path)
    full_features = data["embeddings"]
    full_labels = data["labels"]

    # Deterministic subset extraction
    X_train, y_train = get_first_k_subset(
        full_features, 
        full_labels, 
        shots_per_class=SHOTS_PER_CLASS
    )

    # Train model
    model, final_loss, final_acc = train_linear_model(
        features=X_train,
        labels=y_train,
        num_classes=NUM_CLASSES,
        epochs=NUM_EPOCHS,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    # Save PyTorch state dictionary
    weights_path = os.path.join(weights_output_dir, f"{model_name}_linear.pt")
    torch.save({
        "model_state_dict": model.state_dict(),
        "input_dim": int(X_train.shape[1]),
        "num_classes": NUM_CLASSES,
        "shots_per_class": SHOTS_PER_CLASS,
        "weight_decay": WEIGHT_DECAY,
        "final_loss": final_loss,
        "final_acc": final_acc
    }, weights_path)

    training_summary[model_name] = {
        "shots_per_class": SHOTS_PER_CLASS,
        "total_train_samples": int(X_train.shape[0]),
        "lambda_regularization": WEIGHT_DECAY,
        "epochs": NUM_EPOCHS,
        "final_train_loss": float(final_loss),
        "final_train_accuracy": float(final_acc),
        "weights_path": weights_path
    }

# Save metadata summary
summary_file = os.path.join(weights_output_dir, "training_summary.json")
with open(summary_file, "w") as f:
    json.dump(training_summary, f, indent=4)

print(f"\nAll models trained and weights saved to {weights_output_dir}")

Training Linear Model for AVES-bio (10-shot, λ=0.05)...
Training Linear Model for BirdAVES-biox-base (10-shot, λ=0.05)...
Training Linear Model for DistilHuBERT (10-shot, λ=0.05)...
Training Linear Model for HuBERT (10-shot, λ=0.05)...
Training Linear Model for MusicHuBERT (10-shot, λ=0.05)...
Training Linear Model for WavLM (10-shot, λ=0.05)...

All models trained and weights saved to ../trained_models_10_shot/Linear


### Training the classifier with optimal lamnbda using Cross Validation

##### Only for K = 5, 10, 25

In [3]:
import os
import json
import torch
import torch.nn as nn
import numpy as np

In [6]:

# Deterministic seeding for exact reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

class LinearProbe(nn.Module):
    def __init__(self, input_dim: int, num_classes: int = 6):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

def get_first_k_subset(features: np.ndarray, labels: np.ndarray, shots_per_class: int):
    """
    Deterministically selects the first `shots_per_class` samples for each class
    in the order they appear in the archive.
    """
    selected_indices = []
    for c in np.unique(labels):
        c_indices = np.where(labels == c)[0]
        if len(c_indices) < shots_per_class:
            raise ValueError(f"Class {c} has {len(c_indices)} samples; cannot pick {shots_per_class}.")
        selected_indices.extend(c_indices[:shots_per_class])

    selected_indices = np.array(selected_indices)
    return features[selected_indices], labels[selected_indices]

def generate_stratified_folds(labels: np.ndarray, leave_out_per_class: int = 1):
    """
    Partitions the dataset into V stratified folds, holding out `leave_out_per_class`
    samples per class in each validation fold.
    """
    unique_classes = np.unique(labels)
    class_indices = {c: np.where(labels == c)[0] for c in unique_classes}
    shots_per_class = len(class_indices[unique_classes[0]])
    
    num_folds = shots_per_class // leave_out_per_class
    if num_folds < 2:
        raise ValueError(
            f"Cannot perform CV with {shots_per_class} shots and leave_out_per_class={leave_out_per_class}. "
            f"Need at least 2 folds."
        )

    folds = []
    for f in range(num_folds):
        val_idx = []
        train_idx = []
        start = f * leave_out_per_class
        end = start + leave_out_per_class
        
        for c in unique_classes:
            indices = class_indices[c]
            val_idx.extend(indices[start:end])
            train_idx.extend(np.concatenate([indices[:start], indices[end:]]))
            
        folds.append((np.array(train_idx), np.array(val_idx)))

    return folds

def train_linear_probe(
    X_train: torch.Tensor, 
    y_train: torch.Tensor, 
    input_dim: int, 
    num_classes: int, 
    weight_decay: float, 
    epochs: int = 80, 
    lr: float = 0.01,
    seed: int = SEED
):
    """
    Trains a linear probe via full-batch gradient descent with explicit weight decay.
    """
    torch.manual_seed(seed)
    model = LinearProbe(input_dim=input_dim, num_classes=num_classes).to(X_train.device)
    model.train()

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    for _ in range(epochs):
        optimizer.zero_grad()
        logits = model(X_train)
        loss = criterion(logits, y_train)
        loss.backward()
        optimizer.step()

    return model

def evaluate_loss_and_acc(model: nn.Module, X: torch.Tensor, y: torch.Tensor):
    """
    Computes validation cross-entropy loss and accuracy in eval mode.
    """
    model.eval()
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        logits = model(X)
        loss = criterion(logits, y).item()
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean().item()
    return loss, acc

def tune_lambda_cv(
    features_norm: torch.Tensor,
    labels_tensor: torch.Tensor,
    input_dim: int,
    num_classes: int,
    candidate_lambdas: list,
    leave_out_per_class: int,
    epochs: int = 80,
    lr: float = 0.01
):
    """
    Runs stratified cross-validation strictly within K-shot data across candidate lambdas.
    Returns optimal lambda and the mean CV metrics.
    """
    labels_np = labels_tensor.cpu().numpy()
    folds = generate_stratified_folds(labels_np, leave_out_per_class=leave_out_per_class)
    
    cv_results = {}
    best_lambda = None
    lowest_cv_loss = float("inf")

    for lmbda in candidate_lambdas:
        fold_losses = []
        fold_accs = []
        
        for fold_idx, (train_idx, val_idx) in enumerate(folds):
            X_tr, y_tr = features_norm[train_idx], labels_tensor[train_idx]
            X_va, y_va = features_norm[val_idx], labels_tensor[val_idx]
            
            # Train on training partition of the fold with candidate lambda
            fold_model = train_linear_probe(
                X_train=X_tr,
                y_train=y_tr,
                input_dim=input_dim,
                num_classes=num_classes,
                weight_decay=lmbda,
                epochs=epochs,
                lr=lr,
                seed=SEED + fold_idx
            )
            
            # Evaluate out-of-fold generalization
            val_loss, val_acc = evaluate_loss_and_acc(fold_model, X_va, y_va)
            fold_losses.append(val_loss)
            fold_accs.append(val_acc)

        mean_loss = float(np.mean(fold_losses))
        mean_acc = float(np.mean(fold_accs))
        
        cv_results[str(lmbda)] = {
            "mean_val_loss": mean_loss,
            "mean_val_acc": mean_acc
        }

        # Select lambda minimizing cross-entropy loss (prefer higher lambda on ties)
        if mean_loss <= lowest_cv_loss:
            lowest_cv_loss = mean_loss
            best_lambda = lmbda

    return best_lambda, cv_results

# --- Pipeline Configuration ---
SHOTS_PER_CLASS = 5     # Evaluated on K in {25, 10, 5}
NUM_CLASSES = 6
NUM_EPOCHS = 100
LEARNING_RATE = 0.01

# Tunable samples left out per class for each validation fold
# K=25 -> 5 per class (5-fold) | K=10 -> 2 per class (5-fold) | K=5 -> 1 per class (5-fold)
LEAVE_OUT_MAP = {
    25: 5,
    10: 2,
    5: 1
}
LEAVE_OUT_PER_CLASS = LEAVE_OUT_MAP.get(SHOTS_PER_CLASS, 1)

# CANDIDATE_LAMBDAS = [1e-4, 1e-3, 1e-2, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]
CANDIDATE_LAMBDAS = [1e-4, 1e-3, 1e-2, 0.1, 1.0, 10.0]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Adjust relative paths based on execution from workspace root or inside `models_classification/`[cite: 1]
base_dir = "../model_embeddings" if os.path.exists("../model_embeddings") else "./model_embeddings"[cite: 1]
weights_output_dir = f"../trained_models_{SHOTS_PER_CLASS}_shot/Linear" if os.path.exists("../model_embeddings") else f"./trained_models_{SHOTS_PER_CLASS}_shot/Linear"
os.makedirs(weights_output_dir, exist_ok=True)

training_summary = {}

for model_name in sorted(os.listdir(base_dir)):
    model_path = os.path.join(base_dir, model_name)
    if not os.path.isdir(model_path):
        continue

    npz_path = os.path.join(model_path, "features.npz")
    if not os.path.exists(npz_path):
        continue

    print(f"\n==================================================")
    print(f"Model: {model_name} | {SHOTS_PER_CLASS}-shot | CV Tuning")
    print(f"==================================================")

    data = np.load(npz_path)
    full_features = data["embeddings"]
    full_labels = data["labels"]

    # 1. Deterministic subset extraction (first K samples per class)
    X_train_raw, y_train_raw = get_first_k_subset(
        full_features, 
        full_labels, 
        shots_per_class=SHOTS_PER_CLASS
    )
    
    N, D = X_train_raw.shape

    # 2. Unit-norm feature projection
    norms = np.linalg.norm(X_train_raw, axis=1, keepdims=True) + 1e-8
    features_norm = torch.tensor(X_train_raw / norms, dtype=torch.float32, device=device)
    labels_tensor = torch.tensor(y_train_raw, dtype=torch.long, device=device)

    # 3. In-Sample Stratified Cross-Validation for optimal lambda
    print(f"Running Stratified CV (holding out {LEAVE_OUT_PER_CLASS} sample(s)/class per fold)...")
    best_lambda, cv_grid_results = tune_lambda_cv(
        features_norm=features_norm,
        labels_tensor=labels_tensor,
        input_dim=D,
        num_classes=NUM_CLASSES,
        candidate_lambdas=CANDIDATE_LAMBDAS,
        leave_out_per_class=LEAVE_OUT_PER_CLASS,
        epochs=NUM_EPOCHS,
        lr=LEARNING_RATE
    )
    print(f"Optimal Lambda Selected: {best_lambda} (Validation Loss: {cv_grid_results[str(best_lambda)]['mean_val_loss']:.4f})")

    # 4. Final Retraining on ALL K-shot samples with the optimal lambda
    final_model = train_linear_probe(
        X_train=features_norm,
        y_train=labels_tensor,
        input_dim=D,
        num_classes=NUM_CLASSES,
        weight_decay=best_lambda,
        epochs=NUM_EPOCHS,
        lr=LEARNING_RATE,
        seed=SEED
    )

    final_loss, final_acc = evaluate_loss_and_acc(final_model, features_norm, labels_tensor)
    print(f"Final Retraining on all {N} samples -> Loss: {final_loss:.4f}, Accuracy: {final_acc * 100:.2f}%")

    # 5. Save Model Weights (.pt)
    weights_path = os.path.join(weights_output_dir, f"{model_name}_linear.pt")
    torch.save({
        "model_state_dict": final_model.state_dict(),
        "input_dim": int(D),
        "num_classes": NUM_CLASSES,
        "shots_per_class": SHOTS_PER_CLASS,
        "optimal_lambda": best_lambda,
        "final_train_loss": final_loss,
        "final_train_acc": final_acc,
        "cv_grid_results": cv_grid_results
    }, weights_path)

    training_summary[model_name] = {
        "shots_per_class": SHOTS_PER_CLASS,
        "total_train_samples": int(N),
        "feature_dim": int(D),
        "optimal_lambda": best_lambda,
        "final_train_loss": float(final_loss),
        "final_train_accuracy": float(final_acc),
        "cv_leave_out_per_class": LEAVE_OUT_PER_CLASS,
        "cv_grid_results": cv_grid_results,
        "weights_path": weights_path
    }

# 6. Save Experiment Summary JSON
summary_file = os.path.join(weights_output_dir, "training_summary.json")
with open(summary_file, "w") as f:
    json.dump(training_summary, f, indent=4)

print(f"\nSaved all trained weights and CV summaries to {weights_output_dir}")


Model: AVES-bio | 5-shot | CV Tuning
Running Stratified CV (holding out 1 sample(s)/class per fold)...
Optimal Lambda Selected: 1.0 (Validation Loss: 1.5093)
Final Retraining on all 30 samples -> Loss: 0.4918, Accuracy: 100.00%

Model: BirdAVES-biox-base | 5-shot | CV Tuning
Running Stratified CV (holding out 1 sample(s)/class per fold)...
Optimal Lambda Selected: 0.1 (Validation Loss: 1.4198)
Final Retraining on all 30 samples -> Loss: 0.1968, Accuracy: 100.00%

Model: DistilHuBERT | 5-shot | CV Tuning
Running Stratified CV (holding out 1 sample(s)/class per fold)...
Optimal Lambda Selected: 1.0 (Validation Loss: 1.4788)
Final Retraining on all 30 samples -> Loss: 0.4335, Accuracy: 100.00%

Model: HuBERT | 5-shot | CV Tuning
Running Stratified CV (holding out 1 sample(s)/class per fold)...
Optimal Lambda Selected: 10.0 (Validation Loss: 1.7279)
Final Retraining on all 30 samples -> Loss: 1.4091, Accuracy: 96.67%

Model: MusicHuBERT | 5-shot | CV Tuning
Running Stratified CV (holding 

#### Test on unseen data (eval)

In [9]:
import os
import json
import torch
import torch.nn as nn
import numpy as np

class LinearProbe(nn.Module):
    def __init__(self, input_dim: int, num_classes: int = 6):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

def evaluate_model(model: nn.Module, X: torch.Tensor, y: torch.Tensor, num_classes: int = 6):
    """
    Computes overall accuracy, test loss, and per-class accuracy on evaluation data.
    """
    model.eval()
    criterion = nn.CrossEntropyLoss()
    
    with torch.no_grad():
        logits = model(X)
        loss = criterion(logits, y).item()
        preds = torch.argmax(logits, dim=1)
        
        overall_acc = (preds == y).float().mean().item()
        
        # Per-class accuracy calculation
        class_accs = {}
        for c in range(num_classes):
            mask = (y == c)
            if mask.sum() > 0:
                class_acc = (preds[mask] == y[mask]).float().mean().item()
                class_accs[int(c)] = float(class_acc)
            else:
                class_accs[int(c)] = None

    return overall_acc, loss, class_accs

# --- Configuration & Path Resolution ---
SHOT_SETTINGS = [5, 10, 25]  # Evaluated shot cases
NUM_CLASSES = 6
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Handle execution from repo root or inside `models_classification/`
base_eval_dir = "../model_embeddings_eval" if os.path.exists("../model_embeddings_eval") else "./model_embeddings_eval"
base_trained_dir_prefix = ".." if os.path.exists("../model_embeddings_eval") else "."
results_output_dir = os.path.join(base_trained_dir_prefix, "model_eval_results")
os.makedirs(results_output_dir, exist_ok=True)

all_shot_rankings = {}

for shots in SHOT_SETTINGS:
    # Check potential subfolder structure for trained models[cite: 2]
    trained_dir = os.path.join(base_trained_dir_prefix, f"trained_models_{shots}_shot", "Linear")
    if not os.path.exists(trained_dir):
        trained_dir = os.path.join(base_trained_dir_prefix, f"trained_models_{shots}_shot")
        
    if not os.path.exists(trained_dir):
        print(f"Directory not found for {shots}-shot models: {trained_dir}, skipping.")
        continue

    print(f"\n=======================================================")
    print(f"         EVALUATING LINEAR PROBES: {shots}-SHOT DATA   ")
    print(f"=======================================================")

    shot_results = {}

    for model_name in sorted(os.listdir(base_eval_dir)):
        eval_model_path = os.path.join(base_eval_dir, model_name)
        if not os.path.isdir(eval_model_path):
            continue

        npz_path = os.path.join(eval_model_path, "features.npz")
        weights_path = os.path.join(trained_dir, f"{model_name}_linear.pt")

        if not os.path.exists(npz_path):
            print(f"Missing features.npz for {model_name} in {eval_model_path}")
            continue
        if not os.path.exists(weights_path):
            print(f"Missing checkpoint {weights_path} for {model_name}")
            continue

        # 1. Load evaluation features & labels
        data = np.load(npz_path)
        eval_features = data["embeddings"]
        eval_labels = data["labels"]

        # 2. Apply unit-norm projection (identical to training pipeline)
        norms = np.linalg.norm(eval_features, axis=1, keepdims=True) + 1e-8
        X_test = torch.tensor(eval_features / norms, dtype=torch.float32, device=device)
        y_test = torch.tensor(eval_labels, dtype=torch.long, device=device)

        # 3. Load model checkpoint
        checkpoint = torch.load(weights_path, map_location=device)
        input_dim = checkpoint["input_dim"]
        
        model = LinearProbe(input_dim=input_dim, num_classes=NUM_CLASSES).to(device)
        model.load_state_dict(checkpoint["model_state_dict"])

        # 4. Compute test metrics
        overall_acc, test_loss, class_accs = evaluate_model(model, X_test, y_test, num_classes=NUM_CLASSES)

        shot_results[model_name] = {
            "overall_accuracy": float(overall_acc),
            "test_loss": float(test_loss),
            "optimal_lambda": checkpoint.get("optimal_lambda", None),
            "total_eval_samples": int(X_test.shape[0]),
            "class_wise_accuracy": class_accs
        }

    # 5. Rank models by overall test accuracy (descending)
    ranked_models = sorted(
        shot_results.items(), 
        key=lambda item: item[1]["overall_accuracy"], 
        reverse=True
    )

    print(f"\n--- {shots}-Shot Ranking Summary ---")
    print(f"{'Rank':<5} | {'Model Name':<22} | {'Accuracy':<10} | {'Test Loss':<10} | {'Tuned λ':<8}")
    print("-" * 65)
    
    ranked_dict = {}
    for rank, (name, metrics) in enumerate(ranked_models, start=1):
        tuned_lambda = metrics['optimal_lambda']
        lambda_str = f"{tuned_lambda:.4f}" if isinstance(tuned_lambda, float) else str(tuned_lambda)
        print(f"{rank:<5} | {name:<22} | {metrics['overall_accuracy']*100:>6.2f}%    | {metrics['test_loss']:>8.4f}  | {lambda_str:<8}")
        
        ranked_dict[name] = {
            "rank": rank,
            **metrics
        }

    all_shot_rankings[f"{shots}_shot"] = ranked_dict

    # Save shot-specific evaluation output
    shot_output_file = os.path.join(results_output_dir, f"linear_eval_{shots}_shot.json")
    with open(shot_output_file, "w") as f:
        json.dump(ranked_dict, f, indent=4)

# 6. Save consolidated evaluation JSON across all shot cases
consolidated_file = os.path.join(results_output_dir, "all_linear_eval_rankings.json")
with open(consolidated_file, "w") as f:
    json.dump(all_shot_rankings, f, indent=4)

print(f"\nAll evaluations complete. Consolidated rankings saved to: {consolidated_file}")


         EVALUATING LINEAR PROBES: 5-SHOT DATA   

--- 5-Shot Ranking Summary ---
Rank  | Model Name             | Accuracy   | Test Loss  | Tuned λ 
-----------------------------------------------------------------
1     | AVES-bio               |  63.33%    |   1.2199  | 1.0000  
2     | BirdAVES-biox-base     |  57.33%    |   1.2663  | 0.1000  
3     | MusicHuBERT            |  49.33%    |   1.4820  | 1.0000  
4     | DistilHuBERT           |  47.33%    |   1.3578  | 1.0000  
5     | HuBERT                 |  47.33%    |   1.6385  | 10.0000 
6     | WavLM                  |  42.67%    |   1.5159  | 1.0000  

         EVALUATING LINEAR PROBES: 10-SHOT DATA   

--- 10-Shot Ranking Summary ---
Rank  | Model Name             | Accuracy   | Test Loss  | Tuned λ 
-----------------------------------------------------------------
1     | AVES-bio               |  68.67%    |   1.0132  | 0.0001  
2     | HuBERT                 |  62.67%    |   1.1668  | 0.0001  
3     | BirdAVES-biox-base  

### Train using fixed epochs (No L2 regularization)

In [15]:
import os
import json
import torch
import torch.nn as nn
import numpy as np

# Deterministic seeding for exact reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

class LinearProbe(nn.Module):
    def __init__(self, input_dim: int, num_classes: int = 6):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

def get_first_k_subset(features: np.ndarray, labels: np.ndarray, shots_per_class: int):
    """
    Deterministically selects the first `shots_per_class` samples for each class
    in the order they appear in the archive.
    """
    selected_indices = []
    for c in np.unique(labels):
        c_indices = np.where(labels == c)[0]
        if len(c_indices) < shots_per_class:
            raise ValueError(f"Class {c} has {len(c_indices)} samples; cannot pick {shots_per_class}.")
        selected_indices.extend(c_indices[:shots_per_class])

    selected_indices = np.array(selected_indices)
    return features[selected_indices], labels[selected_indices]

def generate_stratified_folds(labels: np.ndarray, leave_out_per_class: int = 1):
    """
    Partitions the dataset into V stratified folds, holding out `leave_out_per_class`
    samples per class in each validation fold.
    """
    unique_classes = np.unique(labels)
    class_indices = {c: np.where(labels == c)[0] for c in unique_classes}
    shots_per_class = len(class_indices[unique_classes[0]])
    
    num_folds = shots_per_class // leave_out_per_class
    if num_folds < 2:
        raise ValueError(f"Need at least 2 folds for CV. Received shots={shots_per_class}, leave_out={leave_out_per_class}.")

    folds = []
    for f in range(num_folds):
        val_idx = []
        train_idx = []
        start = f * leave_out_per_class
        end = start + leave_out_per_class
        
        for c in unique_classes:
            indices = class_indices[c]
            val_idx.extend(indices[start:end])
            train_idx.extend(np.concatenate([indices[:start], indices[end:]]))
            
        folds.append((np.array(train_idx), np.array(val_idx)))

    return folds

def tune_epochs_cv(
    features_norm: torch.Tensor,
    labels_tensor: torch.Tensor,
    input_dim: int,
    num_classes: int,
    max_epochs: int,
    leave_out_per_class: int,
    lr: float = 0.01
):
    """
    Finds the optimal epoch count using stratified CV with zero explicit regularisation.
    """
    labels_np = labels_tensor.cpu().numpy()
    folds = generate_stratified_folds(labels_np, leave_out_per_class=leave_out_per_class)
    num_folds = len(folds)
    
    # Store validation loss for each epoch across folds: shape (num_folds, max_epochs)
    val_loss_curve = np.zeros((num_folds, max_epochs))
    val_acc_curve = np.zeros((num_folds, max_epochs))
    criterion = nn.CrossEntropyLoss()

    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        X_tr, y_tr = features_norm[train_idx], labels_tensor[train_idx]
        X_va, y_va = features_norm[val_idx], labels_tensor[val_idx]

        torch.manual_seed(SEED + fold_idx)
        model = LinearProbe(input_dim=input_dim, num_classes=num_classes).to(features_norm.device)
        
        # Zero weight decay: optimization relies entirely on epoch early stopping
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)

        for epoch in range(max_epochs):
            model.train()
            optimizer.zero_grad()
            logits = model(X_tr)
            loss = criterion(logits, y_tr)
            loss.backward()
            optimizer.step()

            # Record out-of-fold metrics
            model.eval()
            with torch.no_grad():
                val_logits = model(X_va)
                v_loss = criterion(val_logits, y_va).item()
                v_acc = (torch.argmax(val_logits, dim=1) == y_va).float().mean().item()
                
            val_loss_curve[fold_idx, epoch] = v_loss
            val_acc_curve[fold_idx, epoch] = v_acc

    # Average loss across all folds for each epoch (1-indexed)
    mean_val_loss = np.mean(val_loss_curve, axis=0)
    mean_val_acc = np.mean(val_acc_curve, axis=0)

    # Optimal epoch minimizes mean validation cross-entropy loss
    optimal_epoch = int(np.argmin(mean_val_loss)) + 1
    best_loss = float(mean_val_loss[optimal_epoch - 1])
    best_acc = float(mean_val_acc[optimal_epoch - 1])

    return optimal_epoch, best_loss, best_acc

def train_linear_probe(
    X_train: torch.Tensor, 
    y_train: torch.Tensor, 
    input_dim: int, 
    num_classes: int, 
    epochs: int, 
    lr: float = 0.01
):
    """
    Retrains the linear probe on full K-shot data for exactly `epochs` iterations.
    """
    torch.manual_seed(SEED)
    model = LinearProbe(input_dim=input_dim, num_classes=num_classes).to(X_train.device)
    model.train()

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)

    for _ in range(epochs):
        optimizer.zero_grad()
        logits = model(X_train)
        loss = criterion(logits, y_train)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        final_logits = model(X_train)
        final_loss = criterion(final_logits, y_train).item()
        preds = torch.argmax(final_logits, dim=1)
        final_acc = (preds == y_train).float().mean().item()

    return model, final_loss, final_acc

# --- Pipeline Configuration ---
SHOTS_PER_CLASS = 5     # Set to 25, 10, or 5
NUM_CLASSES = 6
MAX_EPOCHS = 150
LEARNING_RATE = 0.01

LEAVE_OUT_MAP = {
    25: 5,
    10: 2,
    5: 1
}
LEAVE_OUT_PER_CLASS = LEAVE_OUT_MAP.get(SHOTS_PER_CLASS, 1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_dir = "../model_embeddings" if os.path.exists("../model_embeddings") else "./model_embeddings"
weights_output_dir = (
    f"../trained_models_epoch_tuned_{SHOTS_PER_CLASS}_shot/Linear" 
    if os.path.exists("../model_embeddings") 
    else f"./trained_models_epoch_tuned_{SHOTS_PER_CLASS}_shot/Linear"
)
os.makedirs(weights_output_dir, exist_ok=True)

training_summary = {}

for model_name in sorted(os.listdir(base_dir)):
    model_path = os.path.join(base_dir, model_name)
    if not os.path.isdir(model_path):
        continue

    npz_path = os.path.join(model_path, "features.npz")
    if not os.path.exists(npz_path):
        continue

    print(f"\n==================================================")
    print(f"Model: {model_name} | {SHOTS_PER_CLASS}-shot | Epoch CV Tuning")
    print(f"==================================================")

    data = np.load(npz_path)
    full_features = data["embeddings"]
    full_labels = data["labels"]

    # 1. Deterministic subset extraction
    X_train_raw, y_train_raw = get_first_k_subset(
        full_features, 
        full_labels, 
        shots_per_class=SHOTS_PER_CLASS
    )
    N, D = X_train_raw.shape

    # 2. Unit-norm feature projection
    norms = np.linalg.norm(X_train_raw, axis=1, keepdims=True) + 1e-8
    features_norm = torch.tensor(X_train_raw / norms, dtype=torch.float32, device=device)
    labels_tensor = torch.tensor(y_train_raw, dtype=torch.long, device=device)

    # 3. Stratified CV across epochs (0 to MAX_EPOCHS)
    print(f"Searching for optimal epoch in range [1, {MAX_EPOCHS}] (holding out {LEAVE_OUT_PER_CLASS}/class)...")
    optimal_epoch, best_val_loss, best_val_acc = tune_epochs_cv(
        features_norm=features_norm,
        labels_tensor=labels_tensor,
        input_dim=D,
        num_classes=NUM_CLASSES,
        max_epochs=MAX_EPOCHS,
        leave_out_per_class=LEAVE_OUT_PER_CLASS,
        lr=LEARNING_RATE
    )
    print(f"Optimal Epoch Found: {optimal_epoch} (CV Validation Loss: {best_val_loss:.4f}, Accuracy: {best_val_acc*100:.2f}%)")

    # 4. Final Retraining on all K-shot samples for optimal_epoch
    final_model, train_loss, train_acc = train_linear_probe(
        X_train=features_norm,
        y_train=labels_tensor,
        input_dim=D,
        num_classes=NUM_CLASSES,
        epochs=optimal_epoch,
        lr=LEARNING_RATE
    )
    print(f"Retrained on full {N} samples -> Final Train Loss: {train_loss:.4f}, Accuracy: {train_acc*100:.2f}%")

    # 5. Save Model Checkpoint
    weights_path = os.path.join(weights_output_dir, f"{model_name}_linear.pt")
    torch.save({
        "model_state_dict": final_model.state_dict(),
        "input_dim": int(D),
        "num_classes": NUM_CLASSES,
        "shots_per_class": SHOTS_PER_CLASS,
        "optimal_epoch": optimal_epoch,
        "weight_decay": 0.0,
        "final_train_loss": train_loss,
        "final_train_acc": train_acc
    }, weights_path)

    training_summary[model_name] = {
        "shots_per_class": SHOTS_PER_CLASS,
        "total_train_samples": int(N),
        "feature_dim": int(D),
        "optimal_epoch": optimal_epoch,
        "cv_val_loss": best_val_loss,
        "cv_val_acc": best_val_acc,
        "final_train_loss": float(train_loss),
        "final_train_accuracy": float(train_acc),
        "weights_path": weights_path
    }

# 6. Save Run Summary JSON
summary_file = os.path.join(weights_output_dir, "training_summary.json")
with open(summary_file, "w") as f:
    json.dump(training_summary, f, indent=4)

print(f"\nAll models trained and checkpoints saved to {weights_output_dir}")


Model: AVES-bio | 5-shot | Epoch CV Tuning
Searching for optimal epoch in range [1, 150] (holding out 1/class)...
Optimal Epoch Found: 62 (CV Validation Loss: 1.5136, Accuracy: 56.67%)
Retrained on full 30 samples -> Final Train Loss: 0.4565, Accuracy: 100.00%

Model: BirdAVES-biox-base | 5-shot | Epoch CV Tuning
Searching for optimal epoch in range [1, 150] (holding out 1/class)...
Optimal Epoch Found: 90 (CV Validation Loss: 1.4219, Accuracy: 53.33%)
Retrained on full 30 samples -> Final Train Loss: 0.2067, Accuracy: 100.00%

Model: DistilHuBERT | 5-shot | Epoch CV Tuning
Searching for optimal epoch in range [1, 150] (holding out 1/class)...
Optimal Epoch Found: 57 (CV Validation Loss: 1.4745, Accuracy: 53.33%)
Retrained on full 30 samples -> Final Train Loss: 0.4429, Accuracy: 100.00%

Model: HuBERT | 5-shot | Epoch CV Tuning
Searching for optimal epoch in range [1, 150] (holding out 1/class)...
Optimal Epoch Found: 31 (CV Validation Loss: 1.7087, Accuracy: 30.00%)
Retrained on ful

### Test on unseen data (eval)

In [16]:
import os
import json
import torch
import torch.nn as nn
import numpy as np

class LinearProbe(nn.Module):
    def __init__(self, input_dim: int, num_classes: int = 6):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

def evaluate_model(model: nn.Module, X: torch.Tensor, y: torch.Tensor, num_classes: int = 6):
    """
    Computes overall accuracy, test loss, and per-class accuracy on evaluation data.
    """
    model.eval()
    criterion = nn.CrossEntropyLoss()
    
    with torch.no_grad():
        logits = model(X)
        loss = criterion(logits, y).item()
        preds = torch.argmax(logits, dim=1)
        
        overall_acc = (preds == y).float().mean().item()
        
        # Per-class accuracy calculation
        class_accs = {}
        for c in range(num_classes):
            mask = (y == c)
            if mask.sum() > 0:
                class_acc = (preds[mask] == y[mask]).float().mean().item()
                class_accs[int(c)] = float(class_acc)
            else:
                class_accs[int(c)] = None

    return overall_acc, loss, class_accs

# --- Configuration & Path Resolution ---
SHOT_SETTINGS = [5, 10, 25]  # Evaluated shot cases
NUM_CLASSES = 6
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Handle directory resolution across workspace root and subdirectories[cite: 2]
base_eval_dir = "../model_embeddings_eval" if os.path.exists("../model_embeddings_eval") else "./model_embeddings_eval"
base_trained_dir_prefix = ".." if os.path.exists("../model_embeddings_eval") else "."

results_output_dir = os.path.join(base_trained_dir_prefix, "model_eval_results_epoch_tuned")
os.makedirs(results_output_dir, exist_ok=True)

all_shot_rankings = {}

for shots in SHOT_SETTINGS:
    trained_dir = os.path.join(
        base_trained_dir_prefix, 
        f"trained_models_epoch_tuned_{shots}_shot", 
        "Linear"
    )
    
    if not os.path.exists(trained_dir):
        # Fallback to direct shot folder if no 'Linear' subfolder exists
        trained_dir = os.path.join(
            base_trained_dir_prefix, 
            f"trained_models_epoch_tuned_{shots}_shot"
        )
        
    if not os.path.exists(trained_dir):
        print(f"Directory not found for {shots}-shot models: {trained_dir}, skipping.")
        continue

    print(f"\n=======================================================")
    print(f"   EVALUATING EPOCH-TUNED LINEAR PROBES: {shots}-SHOT DATA   ")
    print(f"=======================================================")

    shot_results = {}

    for model_name in sorted(os.listdir(base_eval_dir)):
        eval_model_path = os.path.join(base_eval_dir, model_name)
        if not os.path.isdir(eval_model_path):
            continue

        npz_path = os.path.join(eval_model_path, "features.npz")
        weights_path = os.path.join(trained_dir, f"{model_name}_linear.pt")

        if not os.path.exists(npz_path):
            print(f"Missing features.npz for {model_name} in {eval_model_path}")
            continue
        if not os.path.exists(weights_path):
            print(f"Missing checkpoint {weights_path} for {model_name}")
            continue

        # 1. Load evaluation features & labels
        data = np.load(npz_path)
        eval_features = data["embeddings"]
        eval_labels = data["labels"]

        # 2. Apply unit-norm projection identical to training
        norms = np.linalg.norm(eval_features, axis=1, keepdims=True) + 1e-8
        X_test = torch.tensor(eval_features / norms, dtype=torch.float32, device=device)
        y_test = torch.tensor(eval_labels, dtype=torch.long, device=device)

        # 3. Load model checkpoint
        checkpoint = torch.load(weights_path, map_location=device)
        input_dim = checkpoint["input_dim"]
        
        model = LinearProbe(input_dim=input_dim, num_classes=NUM_CLASSES).to(device)
        model.load_state_dict(checkpoint["model_state_dict"])

        # 4. Evaluate metrics
        overall_acc, test_loss, class_accs = evaluate_model(
            model, 
            X_test, 
            y_test, 
            num_classes=NUM_CLASSES
        )

        shot_results[model_name] = {
            "overall_accuracy": float(overall_acc),
            "test_loss": float(test_loss),
            "optimal_epoch": checkpoint.get("optimal_epoch", None),
            "total_eval_samples": int(X_test.shape[0]),
            "class_wise_accuracy": class_accs
        }

    # 5. Rank models by overall test accuracy (descending)
    ranked_models = sorted(
        shot_results.items(), 
        key=lambda item: item[1]["overall_accuracy"], 
        reverse=True
    )

    print(f"\n--- {shots}-Shot Ranking Summary (Early-Stopped / Epoch-Tuned) ---")
    print(f"{'Rank':<5} | {'Model Name':<22} | {'Accuracy':<10} | {'Test Loss':<10} | {'Epochs':<8}")
    print("-" * 65)
    
    ranked_dict = {}
    for rank, (name, metrics) in enumerate(ranked_models, start=1):
        opt_epoch = metrics["optimal_epoch"]
        epoch_str = str(opt_epoch) if opt_epoch is not None else "N/A"
        print(f"{rank:<5} | {name:<22} | {metrics['overall_accuracy']*100:>6.2f}%    | {metrics['test_loss']:>8.4f}  | {epoch_str:<8}")
        
        ranked_dict[name] = {
            "rank": rank,
            **metrics
        }

    all_shot_rankings[f"{shots}_shot"] = ranked_dict

    # Save shot-specific evaluation JSON
    shot_output_file = os.path.join(results_output_dir, f"linear_eval_{shots}_shot.json")
    with open(shot_output_file, "w") as f:
        json.dump(ranked_dict, f, indent=4)

# 6. Save consolidated evaluation JSON across all shot cases
consolidated_file = os.path.join(results_output_dir, "all_linear_eval_rankings.json")
with open(consolidated_file, "w") as f:
    json.dump(all_shot_rankings, f, indent=4)

print(f"\nAll evaluations complete. Consolidated rankings saved to: {consolidated_file}")


   EVALUATING EPOCH-TUNED LINEAR PROBES: 5-SHOT DATA   

--- 5-Shot Ranking Summary (Early-Stopped / Epoch-Tuned) ---
Rank  | Model Name             | Accuracy   | Test Loss  | Epochs  
-----------------------------------------------------------------
1     | AVES-bio               |  64.00%    |   1.1975  | 62      
2     | BirdAVES-biox-base     |  57.33%    |   1.2687  | 90      
3     | HuBERT                 |  49.33%    |   1.4567  | 31      
4     | MusicHuBERT            |  46.67%    |   1.5335  | 41      
5     | DistilHuBERT           |  45.33%    |   1.3555  | 57      
6     | WavLM                  |  42.67%    |   1.5357  | 46      

   EVALUATING EPOCH-TUNED LINEAR PROBES: 10-SHOT DATA   

--- 10-Shot Ranking Summary (Early-Stopped / Epoch-Tuned) ---
Rank  | Model Name             | Accuracy   | Test Loss  | Epochs  
-----------------------------------------------------------------
1     | AVES-bio               |  67.33%    |   0.9654  | 150     
2     | BirdAVES-biox-b